
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>



<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">

<div style="background: #1B5162; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Lesson 10</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Load Data Incrementally with COPY INTO</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">Use COPY INTO to load data into an existing table, verify its idempotency, and understand incremental ingestion.</div>
</div>

</div>

## REQUIRED — SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before running this notebook, confirm your compute environment at the top-right of the notebook.

- Click the compute dropdown and select **Serverless** (the default option).
- If you do not see Serverless available, contact your workspace administrator.

**Note:** This notebook was developed and tested on **Serverless compute**. Other compute options may work but are not guaranteed to behave the same.
  </div>
</div>

### Setup
Run the cell below to configure your environment.

In [0]:
%run ./Includes/Classroom-Setup-1

**Where we left off:** You created tables with CTAS (code-driven, one-time) and the Upload UI (manual, one-time). Now you'll learn the method used for repeatable, incremental pipelines.


<!-- LEARN: COPY INTO -->
<!-- Template: numbered-steps-with-examples-summary (adapted) -->

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">

<div style="font-size: 20pt; font-weight: 700; color: #0b2026; margin-bottom: 6px;">COPY INTO: Safe, Incremental File Loading</div>
<div style="font-size: 14pt; color: #5A6F77; margin-bottom: 24px;">Unlike CTAS (which creates a new table each time), COPY INTO loads files into an existing table and tracks which files have already been processed. This makes it safe to re-run on a schedule.</div>

<div style="display: flex; flex-direction: column; gap: 16px;">

<!-- Step 1 -->
<div style="background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 18px 20px; position: relative;">
  <div style="position: absolute; top: 0; left: 0; width: 100%; height: 6px; background: #2574B5; border-radius: 8px 8px 0 0;"></div>
  <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 8px;">
    <div style="font-size: 18pt; font-weight: 800; color: #0b2026;">1</div>
    <div style="font-size: 18pt; font-weight: 700; color: #0b2026;">Create an empty table with a defined schema</div>
  </div>
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    COPY INTO loads into an existing table, so you create the table first with the columns and data types you expect.
  </div>
</div>

<!-- Step 2 -->
<div style="background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 18px 20px; position: relative;">
  <div style="position: absolute; top: 0; left: 0; width: 100%; height: 6px; background: #02A36F; border-radius: 8px 8px 0 0;"></div>
  <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 8px;">
    <div style="font-size: 18pt; font-weight: 800; color: #0b2026;">2</div>
    <div style="font-size: 18pt; font-weight: 700; color: #0b2026;">Run COPY INTO to load all files from a directory</div>
  </div>
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    Point COPY INTO at a volume path. It reads every file in that directory and loads the data into your table. It records which files it processed.
  </div>
</div>

<!-- Step 3 -->
<div style="background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 18px 20px; position: relative;">
  <div style="position: absolute; top: 0; left: 0; width: 100%; height: 6px; background: #F8A805; border-radius: 8px 8px 0 0;"></div>
  <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 8px;">
    <div style="font-size: 18pt; font-weight: 800; color: #0b2026;">3</div>
    <div style="font-size: 18pt; font-weight: 700; color: #0b2026;">Re-run safely — only new files get loaded</div>
  </div>
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    If you run COPY INTO again on the same directory, it loads <strong>0 rows</strong> because it already processed those files. If a new file lands in the directory, only that file gets loaded. This is called <strong>idempotency</strong>.
  </div>
</div>

</div>

<!-- Key takeaway -->
<div style="margin-top: 16px; padding: 16px 20px; background: #FFF6F4; border: 3px solid #FF5F46; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <strong>Key takeaway:</strong> COPY INTO is safe to run on a schedule because it never double-loads data. For production workloads at scale, Databricks recommends streaming tables as a more scalable alternative, but the incremental loading concept is the same.
  </div>
</div>

</div>

##### EXPAND FOR ADDITIONAL NOTES

<details>

**How COPY INTO compares to CTAS**

- **CTAS** creates a new table every time. If you run it twice, you get an error (table already exists) or overwrite the table. It's a one-time operation.
- **COPY INTO** loads into an existing table. It tracks which files have been processed using the table's transaction log, so re-running it is safe and only picks up new files.
- Think of CTAS as "create the table from scratch" and COPY INTO as "add new data to the table."

**When to use COPY INTO**

- You have a volume where new files land regularly (daily exports, partner data drops, sensor readings)
- You want to run a scheduled job that picks up new files without reprocessing old ones
- You need a simple, SQL-based incremental loading pattern

**Limitations to know**

- COPY INTO tracks files by path. If you overwrite a file with the same name but different content, COPY INTO won't re-process it.
- For high-volume, low-latency streaming, Databricks recommends streaming tables (Auto Loader under the hood) instead of COPY INTO.

</details>

### Explore: Create an empty table with a defined schema

Unlike CTAS, COPY INTO loads data into an existing table. So first, we create an empty table with the columns and types we expect.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS current_employees_copyinto (
  ID INT,
  FirstName STRING,
  Country STRING,
  Role STRING
);

In [0]:
%sql
SELECT * 
FROM current_employees_copyinto;

The table exists but has **0 rows**. Now let's load data into it.

### Explore: Load data with COPY INTO

COPY INTO reads all CSV files from the volume directory and loads them into the table. Both `employees.csv` (4 rows) and `employees2.csv` (2 rows) are in the directory, so we should get 6 rows total.

In [0]:
result = spark.sql(f"""
    COPY INTO current_employees_copyinto
    FROM '/Volumes/{my_catalog}/{my_schema}/myfiles/'
    FILEFORMAT = CSV
    FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
""")
result.display()

The **num_affected_rows** column should show **6**: four rows from `employees.csv` and two from `employees2.csv`. COPY INTO loaded both files in one pass.

In [0]:
%sql
SELECT * 
FROM current_employees_copyinto;

All 6 employees are in the table.

### Explore: Prove idempotency: re-run the same COPY INTO

What happens if you run the exact same COPY INTO command again? Let's find out.

In [0]:
result = spark.sql(f"""
    COPY INTO current_employees_copyinto
    FROM '/Volumes/{my_catalog}/{my_schema}/myfiles/'
    FILEFORMAT = CSV
    FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
""")
result.display()

**num_affected_rows: 0**. COPY INTO remembered that it already processed both files and skipped them. No duplicate data. This is what makes it safe to run on a schedule. If no new files have arrived, nothing happens.

### Explore: Verify the version history

Let's confirm what COPY INTO recorded in the table's transaction log.

In [0]:
%sql
DESCRIBE HISTORY current_employees_copyinto;

You should see:
- **Version 0** — `CREATE TABLE` (the empty table)
- **Version 1** — `COPY INTO` (loaded 6 rows from 2 files)

Notice there's no version 2 for the second COPY INTO run, since it loaded 0 rows, and no new version was created. The table is unchanged.


<!-- Micro-win summary -->

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">
<div style="margin-top: 10px; padding: 18px 24px; background: #FFF6F4; border: 3px solid #FF5F46; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <div style="font-weight: 700; margin-bottom: 8px;">What you just did:</div>
    <ul style="padding-left: 20px; margin: 0;">
      <li>Created an empty table with a defined schema</li>
      <li>Loaded 6 rows from 2 CSV files using <code>COPY INTO</code></li>
      <li>Proved idempotency by re-running and noting it loaded 0 rows because the files were already processed</li>
      <li>Verified the transaction log only records actual changes</li>
    </ul>
    <div style="margin-top: 12px;">You now know three ingestion methods: <strong>CTAS</strong> (one-time, code-driven), <strong>Upload UI</strong> (one-time, manual), and <strong>COPY INTO</strong> (incremental, schedule-safe).</div>
  </div>
</div>
</div>

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>